In [ ]:
# Example 1: service, file-system, directory and file clients address the right paths
# Building clients is local: nothing below sends a request.
import base64
from azure.storage.filedatalake import (
    DataLakeServiceClient, FileSystemClient, DataLakeDirectoryClient, DataLakeFileClient,
)

key = base64.b64encode(b"not-a-real-account-key").decode()  # a fake shared key
service = DataLakeServiceClient(account_url="https://example.dfs.core.windows.net", credential=key)
assert service.account_name == "example", service.account_name
assert service.primary_hostname == "example.dfs.core.windows.net", service.primary_hostname

file_system = service.get_file_system_client("fs")
directory = file_system.get_directory_client("raw/year=2024")
file_client = directory.get_file_client("logs/app.log")
assert isinstance(file_system, FileSystemClient) and file_system.file_system_name == "fs"
assert isinstance(directory, DataLakeDirectoryClient) and directory.path_name == "raw/year=2024"
assert isinstance(file_client, DataLakeFileClient)
assert file_client.file_system_name == "fs" and file_client.path_name == "raw/year=2024/logs/app.log"
assert file_client.url == "https://example.dfs.core.windows.net/fs/raw/year%3D2024/logs/app.log", file_client.url
# The same file reached straight from the file system has the same URL.
assert file_system.get_file_client("raw/year=2024/logs/app.log").url == file_client.url

# A connection string carries the account name and key.
connection_string = (
    f"DefaultEndpointsProtocol=https;AccountName=example;AccountKey={key};EndpointSuffix=core.windows.net"
)
from_string = DataLakeServiceClient.from_connection_string(connection_string)
assert from_string.url == "https://example.dfs.core.windows.net/", from_string.url
assert from_string.credential.account_name == "example"
print("file system:", file_system.url)
print("directory:  ", directory.url)
print("file:       ", file_client.url)

In [ ]:
# Example 2: directory operations, sent to a local transport that records each request
# RecordingTransport answers every request in-process with an empty success response, so the
# client builds and signs real REST calls without any network access.
from urllib.parse import urlsplit, parse_qs
from azure.core.pipeline.transport import HttpTransport, HttpResponse
from azure.core.utils import CaseInsensitiveDict
from azure.storage.filedatalake import DataLakeServiceClient, DataLakeDirectoryClient


class StubResponse(HttpResponse):
    def __init__(self, request, status_code, headers):
        super().__init__(request, None)
        self.status_code = status_code
        self.headers = CaseInsensitiveDict(headers)
        self.reason = "Stub"
        self.content_type = None

    def body(self):
        return b""


class RecordingTransport(HttpTransport):
    def __init__(self):
        self.requests = []

    def __enter__(self):
        return self

    def __exit__(self, *exc_info):
        pass

    def open(self):
        pass

    def close(self):
        pass

    def send(self, request, **kwargs):
        self.requests.append(request)
        query = parse_qs(urlsplit(request.url).query)
        headers = {"x-ms-request-id": "stub-request"}
        if query.get("comp") == ["lease"]:
            status_code = 201 if request.headers.get("x-ms-lease-action") == "acquire" else 200
            headers["x-ms-lease-id"] = request.headers.get("x-ms-proposed-lease-id", "")
        elif request.method == "PUT":
            status_code = 201
        elif query.get("action") == ["append"]:
            status_code = 202
        else:
            status_code = 200
        return StubResponse(request, status_code, headers)


def query_of(request):
    return {name: values[0] for name, values in parse_qs(urlsplit(request.url).query).items()}


transport = RecordingTransport()
service = DataLakeServiceClient("https://example.dfs.core.windows.net", credential=key, transport=transport)
file_system = service.get_file_system_client("fs")

file_system.create_directory("raw/year=2024/month=01", metadata={"format": "parquet"})
directory = file_system.get_directory_client("raw/year=2024/month=01")
renamed = directory.rename_directory("fs/raw/year=2024/month=02")
assert isinstance(renamed, DataLakeDirectoryClient) and renamed.path_name == "raw/year=2024/month=02"
renamed.delete_directory()

create, rename, delete = transport.requests
assert create.method == "PUT" and query_of(create)["resource"] == "directory", create.url
assert urlsplit(create.url).path == "/fs/raw/year%3D2024/month%3D01", create.url
assert create.headers["Authorization"].startswith("SharedKey example:"), "the request was not signed"
assert rename.method == "PUT" and urlsplit(rename.url).path == "/fs/raw/year%3D2024/month%3D02", rename.url
assert rename.headers["x-ms-rename-source"].startswith("/fs/raw/year%3D2024/month%3D01"), rename.headers
assert delete.method == "DELETE" and urlsplit(delete.url).path == "/fs/raw/year%3D2024/month%3D02", delete.url
for request in transport.requests:
    print(request.method, request.url)

In [ ]:
# Example 3: create a file, append two chunks, then flush them in one commit
# append_data stages bytes at an offset; flush_data commits everything up to a position.
from azure.storage.filedatalake import ContentSettings

transport = RecordingTransport()
service = DataLakeServiceClient("https://example.dfs.core.windows.net", credential=key, transport=transport)
file_client = service.get_file_client("fs", "logs/app.log")

chunks = [b"2024-01-18 10:00:00 INFO started\n", b"2024-01-18 10:00:15 INFO request done\n"]
file_client.create_file(content_settings=ContentSettings(content_type="text/plain"))
offset = 0
for chunk in chunks:
    file_client.append_data(chunk, offset=offset, length=len(chunk))
    offset += len(chunk)
file_client.flush_data(offset)

create, first, second, flush = transport.requests
assert create.method == "PUT" and query_of(create)["resource"] == "file", create.url
assert create.headers["x-ms-content-type"] == "text/plain", create.headers
assert [query_of(r)["action"] for r in (first, second)] == ["append", "append"]
assert [int(query_of(r)["position"]) for r in (first, second)] == [0, len(chunks[0])]
assert [int(r.headers["Content-Length"]) for r in (first, second)] == [len(c) for c in chunks]
assert first.body == chunks[0] and second.body == chunks[1]
assert query_of(flush)["action"] == "flush" and int(query_of(flush)["position"]) == offset == sum(map(len, chunks))
for request in transport.requests:
    print(request.method, request.url)

In [ ]:
# Example 4: POSIX-style access control on a path
# set_access_control sends the ACL or the permission bits as request headers.
transport = RecordingTransport()
service = DataLakeServiceClient("https://example.dfs.core.windows.net", credential=key, transport=transport)
directory = service.get_directory_client("fs", "processed")

acl = "user::rwx,group::r-x,other::---,user:00000000-0000-0000-0000-000000000001:r-x"
directory.set_access_control(acl=acl)
directory.set_access_control(owner="$superuser", group="$superuser", permissions="rwxr-x---")

set_acl, set_permissions = transport.requests
assert set_acl.method == "PATCH" and query_of(set_acl)["action"] == "setAccessControl", set_acl.url
assert set_acl.headers["x-ms-acl"] == acl, set_acl.headers
assert "x-ms-permissions" not in set_acl.headers
assert set_permissions.headers["x-ms-permissions"] == "rwxr-x---", set_permissions.headers
assert set_permissions.headers["x-ms-owner"] == "$superuser"
print("ACL header:        ", set_acl.headers["x-ms-acl"])
print("permissions header:", set_permissions.headers["x-ms-permissions"])

In [ ]:
# Example 5: leases and a file-system SAS token
import datetime
import uuid
from azure.storage.filedatalake import (
    DataLakeLeaseClient, FileSystemSasPermissions, generate_file_system_sas,
)

transport = RecordingTransport()
service = DataLakeServiceClient("https://example.dfs.core.windows.net", credential=key, transport=transport)
file_client = service.get_file_client("fs", "config/settings.json")

proposed = str(uuid.uuid4())
lease = DataLakeLeaseClient(file_client, lease_id=proposed)
lease.acquire(lease_duration=15)
assert lease.id == proposed, (lease.id, proposed)
lease.release()
acquire, release = transport.requests
assert acquire.method == "PUT" and query_of(acquire)["comp"] == "lease", acquire.url
assert acquire.headers["x-ms-lease-action"] == "acquire" and acquire.headers["x-ms-lease-duration"] == "15"
assert acquire.headers["x-ms-proposed-lease-id"] == proposed
assert release.headers["x-ms-lease-action"] == "release" and release.headers["x-ms-lease-id"] == proposed
print("lease requests:", [r.headers["x-ms-lease-action"] for r in transport.requests])

# A SAS token is signed locally with the account key.
permission = FileSystemSasPermissions(read=True, write=True, list=True)
assert str(permission) == "rwl"
assert FileSystemSasPermissions.from_string("rl").write is False
token = generate_file_system_sas(
    "example", "fs", credential=key, permission=permission,
    expiry=datetime.datetime(2030, 1, 1, tzinfo=datetime.timezone.utc),
)
fields = dict(part.split("=", 1) for part in token.split("&"))
assert fields["sp"] == "rwl" and fields["sr"] == "c" and fields["se"] == "2030-01-01T00%3A00%3A00Z", fields
assert fields["sig"], "the SAS token has no signature"
print("SAS fields:", sorted(fields))